# Running Models — Audio · Part 2 — Companion Notebook

This goes with **"Make a Model Talk: Your First Text-to-Speech Run"**. Run cells top to bottom (Shift+Enter). SpeechT5 runs fine on CPU — no GPU needed.

## Install

`sentencepiece` is required by SpeechT5's text processor — don't skip it.

In [ ]:
!pip install -q transformers datasets soundfile sentencepiece

## The easy button — a talking pipeline

SpeechT5 has **no default voice**, so we fetch a 256-D speaker embedding (a 'voice fingerprint') and pass it in.

In [ ]:
from transformers import pipeline
import torch
from datasets import load_dataset

synth = pipeline("text-to-speech", model="microsoft/speecht5_tts")

# a speaker embedding = a 256-D 'voice fingerprint' the model conditions on
embeddings = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker = torch.tensor(embeddings[7306]["xvector"]).unsqueeze(0)

speech = synth(
    "Hello from a model that just learned to talk.",
    forward_params={"speaker_embeddings": speaker},
)
print(speech["sampling_rate"])                 # 16000

Play it right in the notebook.

In [ ]:
from IPython.display import Audio

Audio(speech["audio"], rate=speech["sampling_rate"])

## Save it to a file

The sampling rate has to travel with the samples.

In [ ]:
import soundfile as sf

sf.write("speech.wav", speech["audio"], speech["sampling_rate"])

## The honest way — three parts, by hand

Text-to-speech is an assembly line: processor (tokenize) → model (text encoder → mel spectrogram) → vocoder (spectrogram → waveform). The speaker embedding conditions the spectrogram.

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

inputs = processor(text="The vocoder turns a spectrogram back into sound.", return_tensors="pt")
speech = model.generate_speech(inputs["input_ids"], speaker, vocoder=vocoder)
print(speech.shape)                            # torch.Size([N]) waveform at 16 kHz

`speech` is a PyTorch tensor now, so call `.numpy()` for `Audio`.

In [ ]:
Audio(speech.numpy(), rate=16000)

## Changing the voice

Different index, different voice — same words, same model. The fingerprint does all the work.

In [ ]:
speaker = torch.tensor(embeddings[1000]["xvector"]).unsqueeze(0)

speech = model.generate_speech(inputs["input_ids"], speaker, vocoder=vocoder)
Audio(speech.numpy(), rate=16000)